In [2]:
%pip install dash pandas plotly

  Using cached dash-4.4.1-py3-none-any.whl.metadata (11 kB)
  Using cached retrying-1.4.2-py3-none-any.whl.metadata (5.5 kB)
  Using cached janus-2.0.0-py3-none-any.whl.metadata (5.3 kB)
Using cached dash-4.4.1-py3-none-any.whl (8.9 MB)
Using cached janus-2.0.0-py3-none-any.whl (12 kB)
Using cached retrying-1.4.2-py3-none-any.whl (10 kB)

   -------------------------- ------------- 2/3 [dash]
   -------------------------- ------------- 2/3 [dash]
   -------------------------- ------------- 2/3 [dash]
   -------------------------- ------------- 2/3 [dash]
   -------------------------- ------------- 2/3 [dash]
   -------------------------- ------------- 2/3 [dash]
   -------------------------- ------------- 2/3 [dash]
   -------------------------- ------------- 2/3 [dash]
   -------------------------- ------------- 2/3 [dash]
   -------------------------- ------------- 2/3 [dash]
   -------------------------- ------------- 2/3 [dash]
   -------------------------- ------------- 2/3 [dash]

In [1]:
import pandas as pd
import dash
from dash import dcc, html, Input, Output
import plotly.express as px

# ==========================================
# 1. LOAD DATASETS (Global scope)
# ==========================================
df_geo = pd.read_csv("India_Geospatial.csv")
df_transit = pd.read_csv("India_Transit_Dashboard.csv")

# Get a list of unique cities for the dropdown filter
# We'll combine cities from both datasets just in case, though they likely match
cities = sorted(list(set(df_geo['City'].dropna()).union(set(df_transit['City'].dropna()))))
dropdown_options = [{'label': 'All Cities', 'value': 'All'}] + [{'label': city, 'value': city} for city in cities]

# ==========================================
# 2. BUILD THE DASHBOARD APP & LAYOUT
# ==========================================
app = dash.Dash(__name__)

card_style = {
    'box-shadow': '2px 2px 5px lightgrey', 'border-radius': '5px', 
    'padding': '15px', 'margin': '10px', 'flex': '1', 
    'text-align': 'center', 'background-color': '#f9f9f9',
    'font-family': 'Arial'
}

app.layout = html.Div([
    html.H1("India Transit & Mobility Dashboard", style={'textAlign': 'center', 'font-family': 'Arial'}),
    
    # --- FILTER SECTION ---
    html.Div([
        html.Label("Select City to Filter:", style={'font-weight': 'bold', 'font-family': 'Arial', 'margin-right': '10px'}),
        dcc.Dropdown(
            id='city-filter',
            options=dropdown_options,
            value='All',
            clearable=False,
            style={'width': '300px', 'font-family': 'Arial'}
        )
    ], style={'display': 'flex', 'justify-content': 'center', 'align-items': 'center', 'padding': '20px'}),
    
    # --- MAPS SECTION ---
    html.Div([
        html.Div([dcc.Graph(id='mobility-heatmap')], style={'flex': '1', 'padding': '10px'}),
        html.Div([dcc.Graph(id='transit-flow-map')], style={'flex': '1', 'padding': '10px'}),
    ], style={'display': 'flex', 'flex-direction': 'row'}),
    
    html.Hr(),
    
    # --- KPI SECTION ---
    html.H2("Route Performance Analysis", style={'textAlign': 'center', 'font-family': 'Arial', 'margin-top': '30px'}),
    
    html.Div([
        html.Div([html.H3("Total Passengers"), html.H2(id="kpi-passengers")], style=card_style),
        html.Div([html.H3("Avg Travel Time"), html.H2(id="kpi-travel-time")], style=card_style),
        html.Div([html.H3("Avg Speed"), html.H2(id="kpi-speed")], style=card_style),
        html.Div([html.H3("On-Time Performance"), html.H2(id="kpi-on-time")], style=card_style),
    ], style={'display': 'flex', 'flex-direction': 'row', 'justify-content': 'space-around'}),
    
    # --- CHARTS SECTION ---
    html.Div([
        html.Div([dcc.Graph(id='bar-passengers')], style={'flex': '1', 'padding': '10px'}),
        html.Div([dcc.Graph(id='col-speed')], style={'flex': '1', 'padding': '10px'}),
    ], style={'display': 'flex', 'flex-direction': 'row'}),
])


# ==========================================
# 3. INTERACTIVITY (CALLBACKS)
# ==========================================
@app.callback(
    [Output('mobility-heatmap', 'figure'),
     Output('transit-flow-map', 'figure'),
     Output('kpi-passengers', 'children'),
     Output('kpi-travel-time', 'children'),
     Output('kpi-speed', 'children'),
     Output('kpi-on-time', 'children'),
     Output('bar-passengers', 'figure'),
     Output('col-speed', 'figure')],
    [Input('city-filter', 'value')]
)
def update_dashboard(selected_city):
    # 1. Filter Data based on dropdown selection
    if selected_city == 'All':
        dff_geo = df_geo.copy()
        dff_transit = df_transit.copy()
    else:
        dff_geo = df_geo[df_geo['City'] == selected_city].copy()
        dff_transit = df_transit[df_transit['City'] == selected_city].copy()
        
    dff_transit_source = dff_transit[dff_transit['Point'] == 'Source'].copy()

    # 2. Build Heatmap
    try:
        fig_heatmap = px.density_map(
            dff_geo, lat='Latitude', lon='Longitude', radius=12,
            center=dict(lat=20.5937, lon=78.9629), zoom=3.5,
            map_style="carto-positron", title=f"Mobility Heatmap ({selected_city})"
        )
    except AttributeError:
        fig_heatmap = px.density_mapbox(
            dff_geo, lat='Latitude', lon='Longitude', radius=12,
            center=dict(lat=20.5937, lon=78.9629), zoom=3.5,
            mapbox_style="carto-positron", title=f"Mobility Heatmap ({selected_city})"
        )
    fig_heatmap.update_layout(margin={"r":0,"t":40,"l":0,"b":0})

    # 3. Build Flow Map
    try:
        fig_flow = px.line_map(
            dff_transit, lat='Latitude', lon='Longitude', 
            line_group='Transit_ID', color='Route_ID',
            center=dict(lat=20.5937, lon=78.9629), zoom=3.5,
            map_style="carto-positron", title=f"Transit Flow Map ({selected_city})"
        )
    except AttributeError:
        fig_flow = px.line_mapbox(
            dff_transit, lat='Latitude', lon='Longitude', 
            line_group='Transit_ID', color='Route_ID',
            center=dict(lat=20.5937, lon=78.9629), zoom=3.5,
            mapbox_style="carto-positron", title=f"Transit Flow Map ({selected_city})"
        )
    fig_flow.update_layout(margin={"r":0,"t":40,"l":0,"b":0}, showlegend=False)

    # 4. Calculate KPIs (Handle division by zero if empty)
    if not dff_transit_source.empty:
        total_passengers = dff_transit_source['Passenger_Count'].sum()
        avg_travel_time = dff_transit_source['Travel_Time_Min'].mean()
        avg_speed = dff_transit_source['Traffic_Speed_kmh'].mean()
        on_time_pct = (dff_transit_source['On_Time_Status'] == 'On Time').mean() * 100
    else:
        total_passengers, avg_travel_time, avg_speed, on_time_pct = 0, 0, 0, 0
        
    kpi_1 = f"{total_passengers:,.0f}"
    kpi_2 = f"{avg_travel_time:.1f} min"
    kpi_3 = f"{avg_speed:.1f} km/h"
    kpi_4 = f"{on_time_pct:.1f}%"

    # 5. Build Bar Chart
    df_pass_grp = dff_transit_source.groupby('Route_ID', as_index=False)['Passenger_Count'].sum()
    fig_bar = px.bar(
        df_pass_grp, x='Route_ID', y='Passenger_Count', 
        title='Total Passenger Count by Route',
        color='Passenger_Count', color_continuous_scale='Viridis'
    )

    # 6. Build Column Chart
    df_speed_grp = dff_transit_source.groupby('Route_ID', as_index=False)['Traffic_Speed_kmh'].mean()
    fig_col = px.bar(
        df_speed_grp, x='Route_ID', y='Traffic_Speed_kmh', 
        title='Average Speed by Route',
        color='Traffic_Speed_kmh', color_continuous_scale='Blues'
    )

    # Return elements in the exact order specified in Output()
    return fig_heatmap, fig_flow, kpi_1, kpi_2, kpi_3, kpi_4, fig_bar, fig_col


# ==========================================
# 4. RUN THE APP
# ==========================================
if __name__ == '__main__':
    app.run(jupyter_mode="inline", port=8050)

In [1]:
import pandas as pd
import dash

from dash import dcc, html, Input, Output, State
import plotly.express as px


# ============================================================
# 1. LOAD DATASETS
# ============================================================

df_geo = pd.read_csv("India_Geospatial.csv")
df_transit = pd.read_csv("India_Transit_Dashboard.csv")


# ============================================================
# 2. CLEAN DATA
# ============================================================

# Remove extra spaces from column names
df_geo.columns = df_geo.columns.str.strip()
df_transit.columns = df_transit.columns.str.strip()

# Make sure important numeric columns are numeric
numeric_columns = [
    "Passenger_Count",
    "Travel_Time_Min",
    "Traffic_Speed_kmh",
    "Latitude",
    "Longitude"
]

for col in numeric_columns:
    if col in df_transit.columns:
        df_transit[col] = pd.to_numeric(
            df_transit[col],
            errors="coerce"
        )

if "Latitude" in df_geo.columns:
    df_geo["Latitude"] = pd.to_numeric(
        df_geo["Latitude"],
        errors="coerce"
    )

if "Longitude" in df_geo.columns:
    df_geo["Longitude"] = pd.to_numeric(
        df_geo["Longitude"],
        errors="coerce"
    )


# ============================================================
# 3. CITY DROPDOWN OPTIONS
# ============================================================

cities_geo = set(
    df_geo["City"].dropna().astype(str).unique()
) if "City" in df_geo.columns else set()

cities_transit = set(
    df_transit["City"].dropna().astype(str).unique()
) if "City" in df_transit.columns else set()

cities = sorted(
    cities_geo.union(cities_transit)
)

dropdown_options = [
    {
        "label": "All Cities",
        "value": "All"
    }
]

dropdown_options += [
    {
        "label": city,
        "value": city
    }
    for city in cities
]


# ============================================================
# 4. DASH APP
# ============================================================

app = dash.Dash(__name__)

app.title = "India Transit Dashboard"


# ============================================================
# 5. STYLES
# ============================================================

card_style = {
    "boxShadow": "2px 2px 8px lightgrey",
    "borderRadius": "10px",
    "padding": "15px",
    "margin": "10px",
    "flex": "1",
    "textAlign": "center",
    "backgroundColor": "white",
    "fontFamily": "Arial"
}


kpi_title_style = {
    "fontFamily": "Arial",
    "fontSize": "17px",
    "color": "#555555",
    "marginBottom": "10px"
}


kpi_value_style = {
    "fontFamily": "Arial",
    "fontSize": "30px",
    "fontWeight": "bold",
    "color": "#2c3e50"
}


section_title_style = {
    "textAlign": "center",
    "fontFamily": "Arial",
    "marginTop": "30px"
}


# ============================================================
# 6. DASHBOARD LAYOUT
# ============================================================

app.layout = html.Div(

    [

        # ----------------------------------------------------
        # TITLE
        # ----------------------------------------------------

        html.H1(
            "India Transit & Mobility Dashboard",
            style={
                "textAlign": "center",
                "fontFamily": "Arial",
                "color": "#2c3e50",
                "marginTop": "20px"
            }
        ),


        # ----------------------------------------------------
        # CITY FILTER
        # ----------------------------------------------------

        html.Div(

            [

                html.Label(
                    "Select City:",
                    style={
                        "fontWeight": "bold",
                        "fontFamily": "Arial",
                        "marginRight": "10px"
                    }
                ),

                dcc.Dropdown(
                    id="city-filter",
                    options=dropdown_options,
                    value="All",
                    clearable=False,
                    style={
                        "width": "350px",
                        "fontFamily": "Arial"
                    }
                )

            ],

            style={
                "display": "flex",
                "justifyContent": "center",
                "alignItems": "center",
                "padding": "15px"
            }

        ),


        # ----------------------------------------------------
        # DOWNLOAD BUTTONS
        # ----------------------------------------------------

        html.Div(

            [

                html.Button(
                    "⬇ Download Dashboard",
                    id="download-dashboard-btn",
                    n_clicks=0,
                    style={
                        "backgroundColor": "#2c3e50",
                        "color": "white",
                        "border": "none",
                        "padding": "12px 22px",
                        "borderRadius": "6px",
                        "fontSize": "15px",
                        "cursor": "pointer",
                        "marginRight": "10px",
                        "fontFamily": "Arial"
                    }
                ),

                html.Button(
                    "⬇ Download CSV",
                    id="download-csv-btn",
                    n_clicks=0,
                    style={
                        "backgroundColor": "#27ae60",
                        "color": "white",
                        "border": "none",
                        "padding": "12px 22px",
                        "borderRadius": "6px",
                        "fontSize": "15px",
                        "cursor": "pointer",
                        "fontFamily": "Arial"
                    }
                ),

                dcc.Download(
                    id="download-dashboard"
                ),

                dcc.Download(
                    id="download-csv"
                )

            ],

            style={
                "textAlign": "center",
                "padding": "10px"
            }

        ),


        # ----------------------------------------------------
        # MAP SECTION
        # ----------------------------------------------------

        html.Div(

            [

                html.Div(
                    [
                        dcc.Graph(
                            id="mobility-heatmap"
                        )
                    ],

                    style={
                        "flex": "1",
                        "padding": "10px"
                    }
                ),

                html.Div(
                    [
                        dcc.Graph(
                            id="transit-flow-map"
                        )
                    ],

                    style={
                        "flex": "1",
                        "padding": "10px"
                    }
                )

            ],

            style={
                "display": "flex",
                "flexDirection": "row",
                "flexWrap": "wrap"
            }

        ),


        html.Hr(),


        # ----------------------------------------------------
        # KPI SECTION
        # ----------------------------------------------------

        html.H2(
            "Route Performance Analysis",
            style=section_title_style
        ),


        html.Div(

            [

                # KPI 1
                html.Div(
                    [
                        html.H3(
                            "Total Passengers",
                            style=kpi_title_style
                        ),

                        html.H2(
                            id="kpi-passengers",
                            style=kpi_value_style
                        )
                    ],

                    style=card_style
                ),


                # KPI 2
                html.Div(
                    [
                        html.H3(
                            "Average Travel Time",
                            style=kpi_title_style
                        ),

                        html.H2(
                            id="kpi-travel-time",
                            style=kpi_value_style
                        )
                    ],

                    style=card_style
                ),


                # KPI 3
                html.Div(
                    [
                        html.H3(
                            "Average Speed",
                            style=kpi_title_style
                        ),

                        html.H2(
                            id="kpi-speed",
                            style=kpi_value_style
                        )
                    ],

                    style=card_style
                ),


                # KPI 4
                html.Div(
                    [
                        html.H3(
                            "On-Time Performance",
                            style=kpi_title_style
                        ),

                        html.H2(
                            id="kpi-on-time",
                            style=kpi_value_style
                        )
                    ],

                    style=card_style
                )

            ],

            style={
                "display": "flex",
                "flexDirection": "row",
                "justifyContent": "space-around",
                "flexWrap": "wrap"
            }

        ),


        # ----------------------------------------------------
        # ROUTE CHARTS
        # ----------------------------------------------------

        html.Div(

            [

                html.Div(
                    [
                        dcc.Graph(
                            id="bar-passengers"
                        )
                    ],

                    style={
                        "flex": "1",
                        "padding": "10px",
                        "minWidth": "400px"
                    }
                ),

                html.Div(
                    [
                        dcc.Graph(
                            id="col-speed"
                        )
                    ],

                    style={
                        "flex": "1",
                        "padding": "10px",
                        "minWidth": "400px"
                    }
                )

            ],

            style={
                "display": "flex",
                "flexDirection": "row",
                "flexWrap": "wrap"
            }

        ),


        html.Hr(),


        # ----------------------------------------------------
        # GEOGRAPHIC HIERARCHY
        # ----------------------------------------------------

        html.H2(
            "Passenger Demand by Area",
            style=section_title_style
        ),


        html.P(
            "Explore passenger demand by State, City and Source area.",
            style={
                "textAlign": "center",
                "fontFamily": "Arial",
                "color": "gray"
            }
        ),


        # ----------------------------------------------------
        # TREEMAP + SUNBURST
        # ----------------------------------------------------

        html.Div(

            [

                html.Div(
                    [
                        dcc.Graph(
                            id="geo-hierarchy-treemap"
                        )
                    ],

                    style={
                        "flex": "1",
                        "padding": "10px",
                        "minWidth": "400px"
                    }
                ),

                html.Div(
                    [
                        dcc.Graph(
                            id="geo-hierarchy-sunburst"
                        )
                    ],

                    style={
                        "flex": "1",
                        "padding": "10px",
                        "minWidth": "400px"
                    }
                )

            ],

            style={
                "display": "flex",
                "flexDirection": "row",
                "flexWrap": "wrap"
            }

        ),


        # ----------------------------------------------------
        # TOP 10 AREAS
        # ----------------------------------------------------

        html.Div(
            [
                dcc.Graph(
                    id="top-10-areas-bar"
                )
            ],

            style={
                "padding": "10px"
            }
        )

    ],

    style={
        "backgroundColor": "#f5f6fa",
        "minHeight": "100vh",
        "padding": "10px"
    }

)


# ============================================================
# 7. MAIN DASHBOARD CALLBACK
# ============================================================

@app.callback(

    [

        Output(
            "mobility-heatmap",
            "figure"
        ),

        Output(
            "transit-flow-map",
            "figure"
        ),

        Output(
            "kpi-passengers",
            "children"
        ),

        Output(
            "kpi-travel-time",
            "children"
        ),

        Output(
            "kpi-speed",
            "children"
        ),

        Output(
            "kpi-on-time",
            "children"
        ),

        Output(
            "bar-passengers",
            "figure"
        ),

        Output(
            "col-speed",
            "figure"
        ),

        Output(
            "geo-hierarchy-treemap",
            "figure"
        ),

        Output(
            "geo-hierarchy-sunburst",
            "figure"
        ),

        Output(
            "top-10-areas-bar",
            "figure"
        )

    ],

    [
        Input(
            "city-filter",
            "value"
        )
    ]

)


def update_dashboard(selected_city):

    # ========================================================
    # FILTER DATA
    # ========================================================

    if selected_city == "All":

        dff_geo = df_geo.copy()

        dff_transit = df_transit.copy()

    else:

        dff_geo = df_geo[
            df_geo["City"] == selected_city
        ].copy()

        dff_transit = df_transit[
            df_transit["City"] == selected_city
        ].copy()


    # ========================================================
    # SOURCE DATA
    # ========================================================

    if "Point" in dff_transit.columns:

        dff_transit_source = dff_transit[
            dff_transit["Point"] == "Source"
        ].copy()

    else:

        dff_transit_source = dff_transit.copy()


    # ========================================================
    # 1. MOBILITY HEATMAP
    # ========================================================

    if not dff_geo.empty:

        try:

            fig_heatmap = px.density_map(
                dff_geo,
                lat="Latitude",
                lon="Longitude",
                radius=12,
                center={
                    "lat": 20.5937,
                    "lon": 78.9629
                },
                zoom=3.5,
                map_style="carto-positron",
                title=f"Mobility Heatmap ({selected_city})"
            )

        except AttributeError:

            fig_heatmap = px.density_mapbox(
                dff_geo,
                lat="Latitude",
                lon="Longitude",
                radius=12,
                center={
                    "lat": 20.5937,
                    "lon": 78.9629
                },
                zoom=3.5,
                mapbox_style="carto-positron",
                title=f"Mobility Heatmap ({selected_city})"
            )

    else:

        fig_heatmap = px.scatter_map(
            title="No geographic data available"
        )


    fig_heatmap.update_layout(
        margin={
            "r": 0,
            "t": 50,
            "l": 0,
            "b": 0
        }
    )


    # ========================================================
    # 2. TRANSIT FLOW MAP
    # ========================================================

    if not dff_transit.empty:

        try:

            fig_flow = px.line_map(
                dff_transit,
                lat="Latitude",
                lon="Longitude",
                line_group="Transit_ID",
                color="Route_ID",
                center={
                    "lat": 20.5937,
                    "lon": 78.9629
                },
                zoom=3.5,
                map_style="carto-positron",
                title=f"Transit Flow Map ({selected_city})"
            )

        except AttributeError:

            fig_flow = px.line_mapbox(
                dff_transit,
                lat="Latitude",
                lon="Longitude",
                line_group="Transit_ID",
                color="Route_ID",
                center={
                    "lat": 20.5937,
                    "lon": 78.9629
                },
                zoom=3.5,
                mapbox_style="carto-positron",
                title=f"Transit Flow Map ({selected_city})"
            )

    else:

        fig_flow = px.scatter_map(
            title="No transit data available"
        )


    fig_flow.update_layout(
        margin={
            "r": 0,
            "t": 50,
            "l": 0,
            "b": 0
        },
        showlegend=False
    )


    # ========================================================
    # 3. KPI CALCULATIONS
    # ========================================================

    if not dff_transit_source.empty:

        total_passengers = (
            dff_transit_source["Passenger_Count"]
            .sum()
        )

        avg_travel_time = (
            dff_transit_source["Travel_Time_Min"]
            .mean()
        )

        avg_speed = (
            dff_transit_source["Traffic_Speed_kmh"]
            .mean()
        )

        if "On_Time_Status" in dff_transit_source.columns:

            on_time_pct = (
                dff_transit_source["On_Time_Status"]
                .eq("On Time")
                .mean()
                * 100
            )

        elif "On Time" in dff_transit_source.columns:

            on_time_pct = (
                pd.to_numeric(
                    dff_transit_source["On Time"],
                    errors="coerce"
                )
                .mean()
                * 100
            )

        else:

            on_time_pct = 0

    else:

        total_passengers = 0
        avg_travel_time = 0
        avg_speed = 0
        on_time_pct = 0


    kpi_1 = f"{total_passengers:,.0f}"

    kpi_2 = f"{avg_travel_time:.1f} min"

    kpi_3 = f"{avg_speed:.1f} km/h"

    kpi_4 = f"{on_time_pct:.1f}%"


    # ========================================================
    # 4. PASSENGER COUNT BY ROUTE
    # ========================================================

    if not dff_transit_source.empty:

        df_pass_grp = (
            dff_transit_source
            .groupby(
                "Route_ID",
                as_index=False
            )["Passenger_Count"]
            .sum()
            .nlargest(
                10,
                "Passenger_Count"
            )
        )

        fig_bar = px.bar(

            df_pass_grp,

            x="Route_ID",

            y="Passenger_Count",

            title="Top 10 Routes by Passenger Count",

            labels={
                "Route_ID": "Route",
                "Passenger_Count": "Passenger Count"
            },

            color="Passenger_Count",

            color_continuous_scale="Viridis"

        )

        fig_bar.update_layout(
            xaxis_title="Route",
            yaxis_title="Passenger Count",
            margin={
                "t": 50,
                "l": 40,
                "r": 20,
                "b": 40
            }
        )

    else:

        fig_bar = px.bar(
            title="No passenger data available"
        )


    # ========================================================
    # 5. AVERAGE SPEED BY ROUTE
    # ========================================================

    if not dff_transit_source.empty:

        df_speed_grp = (
            dff_transit_source
            .groupby(
                "Route_ID",
                as_index=False
            )["Traffic_Speed_kmh"]
            .mean()
            .nlargest(
                10,
                "Traffic_Speed_kmh"
            )
        )

        fig_col = px.bar(

            df_speed_grp,

            x="Route_ID",

            y="Traffic_Speed_kmh",

            title="Top 10 Routes by Average Speed",

            labels={
                "Route_ID": "Route",
                "Traffic_Speed_kmh": "Average Speed (km/h)"
            },

            color="Traffic_Speed_kmh",

            color_continuous_scale="Blues"

        )

        fig_col.update_layout(
            xaxis_title="Route",
            yaxis_title="Average Speed (km/h)",
            margin={
                "t": 50,
                "l": 40,
                "r": 20,
                "b": 40
            }
        )

    else:

        fig_col = px.bar(
            title="No speed data available"
        )


    # ========================================================
    # 6. GEOGRAPHIC HIERARCHY
    # ========================================================

    if (
        not dff_transit_source.empty
        and all(
            col in dff_transit_source.columns
            for col in [
                "State",
                "City",
                "Source",
                "Passenger_Count"
            ]
        )
    ):

        df_tree = (

            dff_transit_source

            .groupby(
                [
                    "State",
                    "City",
                    "Source"
                ],
                as_index=False
            )

            ["Passenger_Count"]

            .sum()

        )


        # ====================================================
        # TREEMAP
        # ====================================================

        fig_treemap = px.treemap(

            df_tree,

            path=[
                px.Constant("India"),
                "State",
                "City",
                "Source"
            ],

            values="Passenger_Count",

            title="Passenger Demand Treemap",

            color="Passenger_Count",

            color_continuous_scale="Sunset"

        )

        fig_treemap.update_traces(
            root_color="lightgrey"
        )

        fig_treemap.update_layout(
            margin={
                "t": 50,
                "l": 10,
                "r": 10,
                "b": 10
            }
        )


        # ====================================================
        # SUNBURST
        # ====================================================

        fig_sunburst = px.sunburst(

            df_tree,

            path=[
                "State",
                "City",
                "Source"
            ],

            values="Passenger_Count",

            title="Passenger Demand Sunburst",

            color="Passenger_Count",

            color_continuous_scale="Sunset"

        )

        fig_sunburst.update_layout(
            margin={
                "t": 50,
                "l": 10,
                "r": 10,
                "b": 10
            }
        )


        # ====================================================
        # TOP 10 AREAS
        # ====================================================

        df_top_10 = (

            df_tree

            .groupby(
                "Source",
                as_index=False
            )["Passenger_Count"]

            .sum()

            .nlargest(
                10,
                "Passenger_Count"
            )

        )


        fig_top10 = px.bar(

            df_top_10,

            x="Passenger_Count",

            y="Source",

            orientation="h",

            title="Top 10 Highest Demand Areas",

            color="Passenger_Count",

            color_continuous_scale="Inferno"

        )

        fig_top10.update_layout(

            yaxis={
                "categoryorder":
                "total ascending"
            },

            xaxis_title="Passenger Count",

            yaxis_title="Area"

        )

    else:

        fig_treemap = px.treemap(
            title="No geographic hierarchy data available"
        )

        fig_sunburst = px.sunburst(
            title="No geographic hierarchy data available"
        )

        fig_top10 = px.bar(
            title="No demand data available"
        )


    # ========================================================
    # RETURN ALL OUTPUTS
    # ========================================================

    return (

        fig_heatmap,
        fig_flow,
        kpi_1,
        kpi_2,
        kpi_3,
        kpi_4,
        fig_bar,
        fig_col,
        fig_treemap,
        fig_sunburst,
        fig_top10

    )


# ============================================================
# 8. DOWNLOAD DASHBOARD AS HTML
# ============================================================

@app.callback(

    Output(
        "download-dashboard",
        "data"
    ),

    Input(
        "download-dashboard-btn",
        "n_clicks"
    ),

    State(
        "city-filter",
        "value"
    ),

    prevent_initial_call=True

)


def download_dashboard(
    n_clicks,
    selected_city
):

    # --------------------------------------------------------
    # FILTER DATA
    # --------------------------------------------------------

    if selected_city == "All":

        dff_transit = df_transit.copy()

    else:

        dff_transit = df_transit[
            df_transit["City"] == selected_city
        ].copy()


    # --------------------------------------------------------
    # SOURCE DATA
    # --------------------------------------------------------

    if "Point" in dff_transit.columns:

        dff_source = dff_transit[
            dff_transit["Point"] == "Source"
        ].copy()

    else:

        dff_source = dff_transit.copy()


    # --------------------------------------------------------
    # KPI CALCULATIONS
    # --------------------------------------------------------

    if not dff_source.empty:

        total_passengers = (
            dff_source["Passenger_Count"]
            .sum()
        )

        avg_travel_time = (
            dff_source["Travel_Time_Min"]
            .mean()
        )

        avg_speed = (
            dff_source["Traffic_Speed_kmh"]
            .mean()
        )

        if "On_Time_Status" in dff_source.columns:

            on_time_pct = (
                dff_source["On_Time_Status"]
                .eq("On Time")
                .mean()
                * 100
            )

        else:

            on_time_pct = 0

    else:

        total_passengers = 0
        avg_travel_time = 0
        avg_speed = 0
        on_time_pct = 0


    # --------------------------------------------------------
    # PASSENGER CHART
    # --------------------------------------------------------

    if not dff_source.empty:

        df_pass = (

            dff_source

            .groupby(
                "Route_ID",
                as_index=False
            )["Passenger_Count"]

            .sum()

            .nlargest(
                10,
                "Passenger_Count"
            )

        )

        fig_bar = px.bar(

            df_pass,

            x="Route_ID",

            y="Passenger_Count",

            title="Top 10 Routes by Passenger Count"

        )

    else:

        fig_bar = px.bar(
            title="No passenger data"
        )


    # --------------------------------------------------------
    # SPEED CHART
    # --------------------------------------------------------

    if not dff_source.empty:

        df_speed = (

            dff_source

            .groupby(
                "Route_ID",
                as_index=False
            )["Traffic_Speed_kmh"]

            .mean()

            .nlargest(
                10,
                "Traffic_Speed_kmh"
            )

        )

        fig_speed = px.bar(

            df_speed,

            x="Route_ID",

            y="Traffic_Speed_kmh",

            title="Top 10 Routes by Average Speed"

        )

    else:

        fig_speed = px.bar(
            title="No speed data"
        )


    # --------------------------------------------------------
    # CONVERT CHARTS TO HTML
    # --------------------------------------------------------

    chart_1 = fig_bar.to_html(
        full_html=False,
        include_plotlyjs="cdn"
    )

    chart_2 = fig_speed.to_html(
        full_html=False,
        include_plotlyjs=False
    )


    # --------------------------------------------------------
    # HTML DASHBOARD
    # --------------------------------------------------------

    html_content = f"""

<!DOCTYPE html>

<html>

<head>

<meta charset="UTF-8">

<title>India Transit Dashboard</title>

<style>

body {{

    font-family: Arial, sans-serif;

    background-color: #f5f6fa;

    margin: 0;

    padding: 30px;

}}

h1 {{

    text-align: center;

    color: #2c3e50;

}}

.city {{

    text-align: center;

    color: #555;

    margin-bottom: 25px;

}}

.kpi-container {{

    display: flex;

    gap: 20px;

    justify-content: center;

    flex-wrap: wrap;

}}

.kpi {{

    background: white;

    padding: 20px;

    width: 210px;

    text-align: center;

    border-radius: 10px;

    box-shadow: 2px 2px 8px #cccccc;

}}

.kpi h3 {{

    color: #555;

}}

.value {{

    font-size: 30px;

    font-weight: bold;

    color: #2c3e50;

}}

.charts {{

    display: flex;

    gap: 20px;

    margin-top: 30px;

}}

.chart {{

    background: white;

    flex: 1;

    padding: 10px;

    border-radius: 10px;

}}

@media(max-width: 900px) {{

    .charts {{

        flex-direction: column;

    }}

}}

</style>

</head>


<body>


<h1>
India Transit & Mobility Dashboard
</h1>


<div class="city">

Selected City:

<strong>
{selected_city}
</strong>

</div>


<div class="kpi-container">


<div class="kpi">

<h3>
Total Passengers
</h3>

<div class="value">

{total_passengers:,.0f}

</div>

</div>


<div class="kpi">

<h3>
Average Travel Time
</h3>

<div class="value">

{avg_travel_time:.1f} min

</div>

</div>


<div class="kpi">

<h3>
Average Speed
</h3>

<div class="value">

{avg_speed:.1f} km/h

</div>

</div>


<div class="kpi">

<h3>
On-Time Performance
</h3>

<div class="value">

{on_time_pct:.1f}%

</div>

</div>


</div>


<div class="charts">

<div class="chart">

{chart_1}

</div>


<div class="chart">

{chart_2}

</div>

</div>


</body>

</html>

"""


    return dcc.send_string(

        html_content,

        f"India_Transit_Dashboard_{selected_city}.html"

    )


# ============================================================
# 9. DOWNLOAD FILTERED CSV
# ============================================================

@app.callback(

    Output(
        "download-csv",
        "data"
    ),

    Input(
        "download-csv-btn",
        "n_clicks"
    ),

    State(
        "city-filter",
        "value"
    ),

    prevent_initial_call=True

)


def download_csv(
    n_clicks,
    selected_city
):

    # --------------------------------------------------------
    # FILTER DATA
    # --------------------------------------------------------

    if selected_city == "All":

        dff = df_transit.copy()

    else:

        dff = df_transit[
            df_transit["City"] == selected_city
        ].copy()


    # --------------------------------------------------------
    # DOWNLOAD
    # --------------------------------------------------------

    return dcc.send_data_frame(

        dff.to_csv,

        f"India_Transit_{selected_city}.csv",

        index=False

    )


# ============================================================
# 10. RUN APPLICATION
# ============================================================

if __name__ == "__main__":

    app.run(
        debug=True,
        port=8050
    )

In [3]:
!pip install python-pptx


   ---------------------------------------- 0/2 [XlsxWriter]
   ---------------------------------------- 0/2 [XlsxWriter]
   -------------------- ------------------- 1/2 [python-pptx]
   -------------------- ------------------- 1/2 [python-pptx]
   -------------------- ------------------- 1/2 [python-pptx]
   -------------------- ------------------- 1/2 [python-pptx]
   ---------------------------------------- 2/2 [python-pptx]

